In [ ]:
subscr_flags = (
    subscr
    .groupby('client_id', as_index=False)
    .agg(
        subscribed_in_campaign=('subscribed_in_campaign_flg', 'max'),
        subscribed_after_view=('subscribed_after_view_flg', 'max')
    )
)

scores = pd.concat(
    [
        march_scores.assign(score_month='Март'),
        april_scores.assign(score_month='Апрель')
    ],
    ignore_index=True
)

scores['score'] = pd.to_numeric(scores['score'], errors='coerce')

merged_scores = scores.merge(
    subscr_flags,
    on='client_id',
    how='left'
)

merged_scores[
    ['subscribed_in_campaign', 'subscribed_after_view']
] = merged_scores[
    ['subscribed_in_campaign', 'subscribed_after_view']
].fillna(0).astype(int)

merged_scores['score_bin'] = pd.cut(
    merged_scores['score'],
    bins=10
)

score_stats = (
    merged_scores
    .groupby(['score_month', 'score_bin'], as_index=False, observed=True)
    .agg(
        avg_score=('score', 'mean'),
        clients_cnt=('client_id', 'nunique'),
        subscribed_cnt=('subscribed_in_campaign', 'sum'),
        subscribed_after_view_cnt=('subscribed_after_view', 'sum')
    )
)

score_stats['subscribed_pct'] = (
    score_stats['subscribed_cnt']
    / score_stats['clients_cnt']
    * 100
).round(2)

score_stats['subscribed_after_view_pct'] = (
    score_stats['subscribed_after_view_cnt']
    / score_stats['clients_cnt']
    * 100
).round(2)

score_stats['score_bin'] = score_stats['score_bin'].apply(
    lambda x: f'{x.left:.2f} - {x.right:.2f}'
)